# Sujet 07 · Santander Transaction
Séance : 01 Nom : ZOGO James Porteur : P4

Étapes assignées : 1.7 : trous 1.11 : cardinalité 1.12 : cibles

CONTEXTE GENERAL DU SUJET :

Une banque a 200 000 clients. Pour 160 000 d’entre eux (train.csv), elle sait si le client a fait une transaction ou non. Pour 40 000 autres (a_decider.csv), elle ne le sait pas. Le projet consiste à deviner, pour ces 40 000 clients, lesquels feront une transaction.

On devine en étudiant les 160 000 cas connus. Un modèle est un programme qui cherche des régularités du type « les clients qui ont cette caractéristique transactent plus souvent ». Il applique ensuite ces régularités aux clients inconnus. Chaque client est décrit par 200 nombres, var_0 à var_199, dont on ignore la signification. Pour la réalisation de l'étape 1.7, il s'agira de compter les lignes strictement identiques en décidant de si on les garde ou si on les retire. Pour cela, on devra afficher le nombre de ces lignes, la décision et la raison de cette décision.

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

SEED = 42
CIBLE = "target"
DOSSIER = Path(".")

print("prêt")

étapes 1.7 : TROUS

# étapes 1.7 : TROUS

dossier = Path(sys.argv[1]) if len(sys.argv) > 1 else Path(".")
tr = pd.read_csv(dossier / "train.csv")
ad = pd.read_csv(dossier / "a_decider.csv")
VARS = [c for c in tr.columns if c.startswith("var_")]

# 1. Un même identifiant apparaît-il deux fois dans une table ?
print("ID_code en double dans train     :", tr["ID_code"].duplicated().sum())
print("ID_code en double dans a_decider :", ad["ID_code"].duplicated().sum())

# 2. Une même ligne de 200 valeurs apparaît-elle deux fois ? (on ignore l'identifiant)
print("lignes identiques dans train     :", tr[VARS].duplicated().sum())
print("lignes identiques dans a_decider :", ad[VARS].duplicated().sum())

# 3. Un client de a_decider est-il aussi dans train ? (risque de fuite : réponse déjà connue)
communs = set(tr["ID_code"]) & set(ad["ID_code"])
print("ID_code présents dans les deux tables :", len(communs))

# 4. Une ligne de a_decider est-elle la copie d'une ligne de train, sous un autre identifiant ?
empreintes_train = set(pd.util.hash_pandas_object(tr[VARS], index=False))
empreintes_ad = pd.util.hash_pandas_object(ad[VARS], index=False)
print("lignes de a_decider copiées de train  :", int(empreintes_ad.isin(empreintes_train).sum()))


# étapes 1.11 : CARDINALITés

# --- A. Comprendre l'AUC sur 6 clients inventés ---------------------------------
reel = [1, 1, 1, 0, 0, 0]                  # 3 clients ont transacté, 3 non
score = [0.9, 0.6, 0.3, 0.7, 0.2, 0.1]     # scores donnés par un modèle imaginaire
paires = list(product([s for s, r in zip(score, reel) if r == 1],
                      [s for s, r in zip(score, reel) if r == 0]))
bien = sum(p > n for p, n in paires) + 0.5 * sum(p == n for p, n in paires)
print(f"A. {len(paires)} paires (un « oui » contre un « non ») ; bien classées : {bien:g}")
print(f"   AUC faite à la main = {bien / len(paires):.4f} | AUC de scikit-learn = {roc_auc_score(reel, score):.4f}\n")

# --- B. Sur les vraies données ---------------------------------------------------
dossier = Path(sys.argv[1]) if len(sys.argv) > 1 else Path(".")
tr = pd.read_csv(dossier / "train.csv")
VARS = [c for c in tr.columns if c.startswith("var_")]
Xa, Xv, ya, yv = train_test_split(tr[VARS], tr["target"], test_size=0.25,
                                  stratify=tr["target"], random_state=42)
taux = yv.mean()
print(f"B. validation : {len(yv):,} clients, {taux:.2%} de positifs")

# Trois « candidats » à noter
Z = (Xv - Xa.mean()) / Xa.std()
modele = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500)).fit(Xa, ya)
candidats = {
    "toujours « non » (constant)": np.zeros(len(yv)),
    "hasard": np.random.RandomState(0).rand(len(yv)),
    "règle simple : moyenne des |z|": Z.abs().mean(axis=1).to_numpy(),
    "régression logistique": modele.predict_proba(Xv)[:, 1],
}
print(f"\n{'candidat':<34}{'exactitude':>11}{'AUC':>8}{'AUC-PR':>9}")
for nom, s in candidats.items():
    decision = (s >= 0.5).astype(int) if nom.startswith("régression") else np.zeros(len(yv), int)
    if nom == "hasard":
        decision = (s >= 0.5).astype(int)
    if nom.startswith("règle"):
        decision = (s >= np.quantile(s, 1 - taux)).astype(int)
    print(f"{nom:<34}{accuracy_score(yv, decision):>11.2%}{roc_auc_score(yv, s):>8.4f}"
          f"{average_precision_score(yv, s):>9.4f}")
print(f"\nRéférence AUC-PR du hasard = taux de positifs = {taux:.4f}")

# étapes 1.12 : CIBLES

COUT_VENTE_MANQUEE = 10.0   # faux négatif : client qui aurait transacté, non sollicité
COUT_SOLLICITATION = 1.0    # faux positif : client sollicité pour rien

dossier = Path(sys.argv[1]) if len(sys.argv) > 1 else Path(".")
tr = pd.read_csv(dossier / "train.csv")
VARS = [c for c in tr.columns if c.startswith("var_")]
Xa, Xv, ya, yv = train_test_split(tr[VARS], tr["target"], test_size=0.25,
                                  stratify=tr["target"], random_state=42)
modele = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500)).fit(Xa, ya)
p = modele.predict_proba(Xv)[:, 1]            # probabilité de transaction, par client
y = yv.to_numpy()
n, n_pos = len(y), int(y.sum())


def cout(decision, c_fn=COUT_VENTE_MANQUEE, c_fp=COUT_SOLLICITATION):
    fn = int(((decision == 0) & (y == 1)).sum())
    fp = int(((decision == 1) & (y == 0)).sum())
    return fn * c_fn + fp * c_fp, fn, fp


# 1. Le seuil théorique : on sollicite si p x C_FN > (1-p) x C_FP
t_star = COUT_SOLLICITATION / (COUT_SOLLICITATION + COUT_VENTE_MANQUEE)
print(f"1. Seuil théorique t* = C_FP/(C_FP+C_FN) = {t_star:.3f} : on contacte si p ≥ {t_star:.3f}\n")

# 2. Comparer des politiques (sur 40 000 clients de validation)
print(f"2. {'politique':<40}{'coût':>9}{'FN':>8}{'FP':>8}{'contactés':>11}")
politiques = {
    "ne contacter personne": np.zeros(n, int),
    "contacter tout le monde": np.ones(n, int),
    f"modèle, seuil t* = {t_star:.3f}": (p >= t_star).astype(int),
    "modèle, seuil 0,5 (défaut)": (p >= 0.5).astype(int),
}
for nom, d in politiques.items():
    c, fn, fp = cout(d)
    print(f"   {nom:<40}{c:>9,.0f}{fn:>8,}{fp:>8,}{d.sum():>11,}")

# 3. Meilleur point de coupe trouvé en balayant les seuils
seuils = np.linspace(0.01, 0.9, 90)
couts = [cout((p >= s).astype(int))[0] for s in seuils]
k = int(np.argmin(couts))
print(f"\n3. Meilleur seuil mesuré : {seuils[k]:.2f} → coût {couts[k]:,.0f}")

# 4. Et si les coûts changent ? (C_FP fixé à 1, on fait varier C_FN)
print(f"\n4. {'C_FN/C_FP':>10}{'meilleur de (personne, tous)':>30}{'modèle (t*)':>14}{'gain':>8}")
for r in [2, 5, 9, 10, 20, 50]:
    base = min(n_pos * r, (n - n_pos) * 1.0)
    t = 1 / (1 + r)
    c_mod = cout((p >= t).astype(int), c_fn=r, c_fp=1.0)[0]
    print(f"{r:>13}{base:>30,.0f}{c_mod:>14,.0f}{1 - c_mod / base:>8.0%}")
print(f"\nSolliciter tout le monde bat « personne » dès que C_FN/C_FP > (1-p)/p = {(1 - y.mean()) / y.mean():.1f}")
